# Tech Addiction Prediction: Logistic Regression Baseline
In this notebook, we build a robust baseline model using Logistic Regression. 
We will implement a 5-Fold Stratified Cross Validation to evaluate the model without overfitting, tune the classification threshold to maximize our F1-score, and finally train a model on the entire dataset to make predictions for our submission.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, classification_report, confusion_matrix,
    precision_recall_curve
)


## 1. Data Loading
We load the data from the standard Kaggle input directory.


In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
SUBMISSION_PATH = '/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv'

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

X = train_df.drop(['id', 'addicted_label'], axis=1)
y = train_df['addicted_label'].values

print(f"Train features shape: {X.shape}")
print(f"Test features shape: {test_df.drop(['id'], axis=1).shape}")


## 2. Preprocessing Pipeline
We need to handle missing values and encode our features properly for a linear model.
- **Numerical:** Impute with median, scale with StandardScaler.
- **Nominal Categorical:** Impute with mode, One-Hot Encode.
- **Ordinal Categorical:** Impute with mode, Ordinal Encode (respecting the low/medium/high hierarchy).


In [ ]:
# Identify column types
numeric_features = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 
    'gaming_hours', 'work_study_hours', 'sleep_hours', 
    'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time'
]

categorical_nominal = ['gender', 'academic_work_impact']
categorical_ordinal = ['stress_level']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Low', 'Medium', 'High']], handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('nom', nominal_transformer, categorical_nominal),
        ('ord', ordinal_transformer, categorical_ordinal)
    ])


## 3. Stratified 5-Fold Cross Validation
We evaluate the model using 5 folds to generate Out-Of-Fold (OOF) predictions. We will then tune our classification threshold over the entire set of OOF predictions to strictly maximize our F1-score.


In [ ]:
print("Starting 5-Fold Stratified Cross-Validation...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Training Fold {fold + 1}/5...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"))
    ])
    
    pipeline.fit(X_train, y_train)
    oof_preds[val_idx] = pipeline.predict_proba(X_val)[:, 1]

print("\nCross-Validation complete!")


### Threshold Tuning & Metric Evaluation


In [ ]:
print("Tuning classification threshold to maximize F1-score...")
precisions, recalls, thresholds = precision_recall_curve(y, oof_preds)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Optimal classification threshold found: {optimal_threshold:.4f}")

# Apply the optimal threshold
y_pred_binary = (oof_preds >= optimal_threshold).astype(int)

# Calculate overall metrics
accuracy = accuracy_score(y, y_pred_binary)
precision = precision_score(y, y_pred_binary)
recall = recall_score(y, y_pred_binary)
f1 = f1_score(y, y_pred_binary)
roc_auc = roc_auc_score(y, oof_preds)
pr_auc = average_precision_score(y, oof_preds)

print("-" * 30)
print("Baseline Model (5-Fold OOF) Performance:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("-" * 30)

print("\nClassification Report:")
print(classification_report(y, y_pred_binary))

print("Confusion Matrix:")
print(confusion_matrix(y, y_pred_binary))


## 4. Train Final Model on Full Dataset
To maximize our predictive power on the test set, we will now retrain the exact same pipeline using **100%** of our training data.


In [ ]:
print("Training final model on full dataset...")
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"))
])

final_pipeline.fit(X, y)
print("Final model trained!")


## 5. Inference and Submission
We predict the probabilities for the unseen test set, apply the `optimal_threshold` we discovered during cross-validation, and write to `submission.csv`.


In [ ]:
# Get test features
X_test = test_df.drop(['id'], axis=1)

# Predict probabilities
test_preds_proba = final_pipeline.predict_proba(X_test)[:, 1]

# Apply optimal threshold
test_preds_binary = (test_preds_proba >= optimal_threshold).astype(int)

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': test_preds_binary
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
display(submission.head())
